# Phase 3b — Approach 2: ResNet-50 Feature Extraction

**CSC3109 Machine Learning | Group 30**

## What Is Transfer Learning?

Instead of learning everything from scratch, we start from a model already trained on **ImageNet** — 1.2 million images across 1,000 categories. This model already knows how to detect edges, textures, shapes, and complex visual patterns.

We then adapt it to our 4-category aerial problem.

## Feature Extraction Strategy

In this approach, we **freeze** all of ResNet-50's layers — their weights are locked and cannot change. We only train the final classification layer (called the **head**), which we replace with our own 4-class version.

```
ResNet-50 (frozen -- 23M params, weights from ImageNet)
   |
   |  extracts 2048-dimensional feature vector
   |
Linear(2048 -> 4)   <-- THIS is the only trainable layer
   |
4 class scores
```

**Why freeze?** The pre-trained features already capture rich visual information. We just teach the final layer: *which combination of those features predicts our 4 categories?*

**Why ResNet-50?** It introduced skip connections (residual connections) that let very deep networks train without the vanishing gradient problem. It is a reliable, well-studied backbone.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT / 'src'))

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models

from dataset import get_data_loaders
from train_utils import train_model, plot_training_curves, plot_confusion_matrix, \
                        get_all_predictions, print_classification_report

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
REPORT_DIR = REPO_ROOT / 'report'

print(f'Device: {DEVICE}')

---
## Step 1 — Load Data

In [ ]:
train_loader, val_loader, cls2idx, idx2cls = get_data_loaders(
    data_dir=REPO_ROOT / 'data', batch_size=32, num_workers=0
)
NUM_CLASSES = len(cls2idx)
print(f'Classes: {cls2idx}')

---
## Step 2 — Load Pre-trained ResNet-50 and Freeze All Layers

In [ ]:
# Load ResNet-50 with ImageNet pre-trained weights
# First time: downloads ~100MB weights file automatically
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

# Freeze ALL layers — no gradients computed for these parameters
for param in model.parameters():
    param.requires_grad = False

# Replace the final classification layer
# ResNet-50's original head is Linear(2048 → 1000) for 1000 ImageNet classes
# We replace it with Linear(2048 → 4) for our 4 categories
in_features = model.fc.in_features  # 2048
model.fc = nn.Linear(in_features, NUM_CLASSES)
# model.fc is NOT frozen (created fresh above — requires_grad=True by default)

# Count trainable vs frozen parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'ResNet-50 loaded.')
print(f'Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')
print('Only the final FC layer (2048 → 4) will be trained.')

---
## Step 3 — Configure Training

Since only the head is trainable, we only pass `model.fc.parameters()` to the optimiser. This is slightly more explicit but functionally the same as passing `model.parameters()` (frozen params are automatically skipped).

In [ ]:
NUM_EPOCHS = 20  # feature extraction converges fast

optimizer = optim.Adam(model.fc.parameters(), lr=1e-3)

scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

print(f'Optimiser: Adam on FC layer only (lr=1e-3)')
print(f'Scheduler: StepLR (reduce by 10x every 7 epochs)')
print(f'Epochs: {NUM_EPOCHS}')
print('Tip: feature extraction is faster than training from scratch — fewer params to update')

---
## Step 4 — Train

In [ ]:
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    num_epochs=NUM_EPOCHS,
    save_name='resnet50_extraction',
    device=DEVICE
)

---
## Step 5 — Plot Training Curves

In [ ]:
plot_training_curves(
    history,
    title='Approach 2 — ResNet-50 Feature Extraction',
    save_path=REPORT_DIR / 'approach2_resnet50_extraction_curves.png'
)

---
## Step 6 — Evaluate

In [ ]:
all_labels, all_preds = get_all_predictions(model, val_loader, DEVICE)

print('=== Approach 2: ResNet-50 Feature Extraction — Validation Results ===')
print_classification_report(all_labels, all_preds)
print(f'Overall Accuracy: {(all_labels == all_preds).mean():.4f}')

In [ ]:
plot_confusion_matrix(
    all_labels, all_preds,
    title='Approach 2 — ResNet-50 Feature Extraction',
    save_path=REPORT_DIR / 'approach2_resnet50_extraction_confusion.png'
)

---
## Step 7 — Observations for Report

**Expected outcome:** Feature extraction should significantly outperform the custom CNN because it leverages features learned from 1.2M images.

**Strengths:**
- Fast to train (only 8,196 parameters trainable)
- Strong generalisation from ImageNet pre-training
- Very low risk of overfitting

**Weaknesses:**
- The frozen features were learned from everyday objects (dogs, cats, cars) — not aerial imagery
- Cannot adapt intermediate features to domain-specific patterns
- Expected to be outperformed by fine-tuning (Approach 3)

*Fill in actual numbers and observations after training.*

---
Best model saved to `models/resnet50_extraction_best.pth`

**Next:** `03c_resnet50_finetune.ipynb` — ResNet-50 Fine-Tuning